In [ ]:
import os
import numpy as np
import pandas as pd

# Define paths and target column
INPUT_DIR = '/kaggle/input/competitions/playground-series-s6e9/'
SUB_PATH = 'megayak/s6e9-0-94645-four-feature-views-beat-the-blend/submission.csv'
NOTEBOOKS_DIR = '/kaggle/input/notebooks/'

TARGET = 'Will_Buy_EV'
FULL_SUB_PATH = os.path.join(NOTEBOOKS_DIR, SUB_PATH)

def apply_power_residual(base_values, reference_values, power):
    """
    Applies sign-preserved power transformation to the residual difference.
    """
    residual = base_values - reference_values
    sign = np.sign(residual)
    powered_residual = sign * (residual.abs() ** power)
    return base_values + powered_residual

def main():
    print("--- Loading baseline submission datasets ---")
    sub_df = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
    
    # Load the base predictions
    raw_predictions = pd.read_csv(FULL_SUB_PATH)[TARGET]
    df_round7 = raw_predictions.round(7)
    df_round6 = raw_predictions.round(6)
    
    print("--- Executing Round 1: Running the stable 0.74 Magic ---")
    magic_power = 0.74 
    round_1_predictions = apply_power_residual(df_round7, df_round6, magic_power)
    
    print("--- Applying Rank Transformation & Post-Blending ---")
    # Convert predictions to percentile ranks to stabilize the distribution
    ranked_predictions = round_1_predictions.rank(pct=True)
    
    # Define blending weights for optimization
    # Feel free to adjust: e.g., 0.95 for less rank impact, or 0.85 for more rank impact
    blend_weight = 0.90
    final_predictions = (blend_weight * round_1_predictions) + ((1 - blend_weight) * ranked_predictions)
    
    # Ensure all predictions stay strictly bounded within valid probability limits
    final_predictions = np.clip(final_predictions, 0.0, 1.0)
    
    # Map predictions back to the submission DataFrame
    sub_df[TARGET] = final_predictions
    
    # Hard integrity check to safeguard against any hidden NaN bugs
    assert not sub_df[TARGET].isna().any(), "Error: NaN values detected in final predictions!"
    
    # Save the optimized submission
    output_filename = 'submission.csv'
    sub_df.to_csv(output_filename, index=False)
    print(f"--- Success! Final stabilized submission saved to '{output_filename}' ---")
    print(sub_df.head(10))

if __name__ == '__main__':
    main()